# Idea

Encoded vector is traited as number. Using mod and div operations we decode length and type of token, and then decode word as mods by 26 or number as mods by 10.

Example of decoding for 128-bit vector:

1. Calculate the base for token prefix:
    - $V$ -- the encoded vector (number till $2^{128}$);
    - $R$ -- is the largest value that can be stored in rest of vector after token prefix;
    - $L_w = log_{26}(R)$ -- is length of longest word in the rest of vector
    - $L_n = log_{10}(R)$ -- is length of longest number in the rest of vector
    - $L_{token} = L_w + L_n$ -- is length of longest token in the rest of vector
    - Formula for $R$ is not obvious: for the first token $R = 2^{128}$ is wrong, because we need to reserve some bits for token prefix. We can try to find $R$ iteratively:
        - $R_0 := 2^{128} => L_{token,0}$
        - $R_{n+1} := R_n / L_{token,n} => L_{token,n+1}$
        - Stop when $L_{token,n+1} = L_{token,n}$
2. Calculate the base by token prefix:
    - $B_{token} := V \mod L_{token}$ -- the length of next token, and base of radix for next token;
    - If $B_{token} = 0$ -- the next token is empty (no matter is it a word or number);
    - If $B_{token} \leq L_w$ -- the next token is word;
    - Else -- the next token is number, $B_{token} := B_{token} - L_w$;
3. Decode the token:
    - $s_i = V \mod B_{token}$ -- the symbol index in token, where $i$ is the position of symbol in token;
    - $V := V \div B_{token}$ -- the rest of vector to continue decoding;
4. Repeat steps 2-3 until $V = 0$.

Example of encoding for id:
1. Split id into tokens;
2. **TODO**

So the $L_{token}$ and related numbers can be precalculated and their calculation should not be part of decoding process.

# Density

In [32]:
import math
import itertools as it;

vector_lengths = [64, 128, 192, 256]

word_alphabet_size = 26 # 'a'-'z'
word_alphabet_size_log2 = math.log2(word_alphabet_size)
number_alphabet_size = 10 # '0'-'9'
number_alphabet_size_log2 = math.log2(number_alphabet_size)

### rest_value is up to 2^256 - 1, so we cannot use math.floor
def get_token_prefix_length(rest_value : int) -> int:
    if rest_value < 1:
        return 0

    rest_value_log2 = rest_value.bit_length() - 1
    longest_word_length = math.floor(rest_value_log2 / word_alphabet_size_log2)
    longest_number_length = math.floor(rest_value_log2 / number_alphabet_size_log2)
    prefix_value_max = longest_word_length + longest_number_length
    prefix_value_last = 0
    while prefix_value_last < prefix_value_max:
        prefix_value_last = prefix_value_max
        rest_value_log2 = (rest_value // (prefix_value_max + 1)).bit_length() - 1
        longest_word_length = math.floor(rest_value_log2 / word_alphabet_size_log2)
        longest_number_length = math.floor(rest_value_log2 / number_alphabet_size_log2)
        prefix_value_max = longest_word_length + longest_number_length
    return prefix_value_max

prefix_bases = [get_token_prefix_length(1 << rest_value) for rest_value in range(vector_lengths[-1] + 1)]

for vector_rest in map(range(16), vector_lengths):
    print(f"Prefix base[{vector_rest}] = {prefix_bases[vector_rest]}")


Prefix base[0] = 0
Prefix base[1] = 0
Prefix base[2] = 0
Prefix base[3] = 0
Prefix base[4] = 0
Prefix base[5] = 0
Prefix base[6] = 1
Prefix base[7] = 2
Prefix base[64] = 29
Prefix base[128] = 61
Prefix base[192] = 94
Prefix base[256] = 126


## Worst case

Each token is single letter word

## Best case

One token should occupies the whole vector. One number token is obviously better, because it has less bits per symbol. But remind that we are encoding identifier, so it should start with letter or separator symbol (like `w23` or `_42`).


In [36]:
for vector_length in vector_lengths:
    prefix_base = prefix_bases[vector_length]
    vector_rest = ((1 << vector_length) - 1) // prefix_base
    vector_rest_log2 = vector_rest.bit_length() - 1
    letters_count = vector_rest_log2 / word_alphabet_size_log2
    numbers_count = vector_rest_log2 / number_alphabet_size_log2
    letters_density = vector_length / letters_count
    numbers_density = vector_length / numbers_count
    print(f"Vector length: {vector_length} bits, prefix base: {prefix_base}, letters count: {letters_count} ({letters_density} bits/letter), numbers count: {numbers_count} ({numbers_density} bits/number)")

Vector length: 64 bits, prefix base: 29, letters count: 12.552017159648427 (5.098782067136099 bits/letter), numbers count: 17.76076974417489 (3.6034474249625625 bits/number)
Vector length: 128 bits, prefix base: 61, letters count: 25.95501853351031 (4.931608884607047 bits/letter), numbers count: 36.725659471005706 (3.4853016077506753 bits/number)
Vector length: 192 bits, prefix base: 94, letters count: 39.35801990737219 (4.878294193962646 bits/letter), numbers count: 55.690549197836525 (3.447622671450668 bits/number)
Vector length: 256 bits, prefix base: 126, letters count: 52.97376733478743 (4.832580593751484 bits/letter), numbers count: 74.95646892033132 (3.4153156316914246 bits/number)



## Average case
